[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mist-medical/MIST/blob/main/examples/mist_nnfoundation_spleen_demo.ipynb)

# Fine-Tuning a Pretrained Foundation Model with MIST — Spleen CT (MSD Task 9)

Training a segmentation network from scratch needs a lot of labeled data. A
**pretrained encoder** gives the network a head start: it has already learned
general-purpose features of 3D medical images from a large unlabeled corpus, so
fine-tuning it on a small labeled dataset can converge faster and generalize
better.

This notebook fine-tunes
[nnFoundationCNN](https://huggingface.co/MIC-DKFZ/nnFoundationCNN), an openly
available ResEnc-L network pretrained with [nnssl](https://github.com/MIC-DKFZ/nnssl)
as a masked autoencoder on 2.1 million CT, MRI, and PET volumes
([Harsy et al., 2026](https://arxiv.org/abs/2609.26924)), on the
[Medical Segmentation Decathlon](http://medicaldecathlon.com/) spleen CT task.
MIST's `resenc` architecture is the same network, so the checkpoint loads
directly with `--pretrained-weights` — no conversion step.

We will walk through:
1. Installing MIST and downloading the data and the pretrained checkpoint
2. Looking inside the checkpoint
3. Converting the dataset, analyzing it, and configuring a run that matches the
   pretrained encoder
4. Training two identical models, one **from scratch** and one **fine-tuned from
   nnFoundationCNN**
5. Comparing the two: learning curves, cross-validation metrics, `mist_rank`, and
   predictions
6. Where to go next: nnFoundationViT and pretraining your own encoder

> **Requirements:** A high-end GPU runtime is strongly recommended
> (`Runtime → Change runtime type`). We developed this notebook on Colab's **G4**
> runtime (NVIDIA Blackwell, 96 GB). The free **T4** is not recommended: its
> two CPU cores take about 30 minutes just to preprocess the data (about 2.5
> minutes on a G4), and training at this patch size is slow.

## Section 1: Setup

Install MIST 2.4.0 or later (the first release with `resenc` and support for
nnssl checkpoints), with NVIDIA DALI for fast GPU data loading.

In [ ]:
!pip install "mist-medical[dali]>=2.4.0" -q

Download the MSD Task 9 (Spleen) dataset: 41 portal-venous CT scans with spleen
labels for training and 20 unlabeled scans for testing (about 1.6 GB).

In [ ]:
!wget -q https://msd-for-monai.s3-us-west-2.amazonaws.com/Task09_Spleen.tar
!tar -xf Task09_Spleen.tar

Download the nnFoundationCNN checkpoint from Hugging Face (about 410 MB). The
weights are released under the
[CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) license; see the
[model card](https://huggingface.co/MIC-DKFZ/nnFoundationCNN) for details and
attribution.

In [ ]:
!wget -q https://huggingface.co/MIC-DKFZ/nnFoundationCNN/resolve/main/checkpoint_final.pth \
    -O nnfoundation_cnn.pth

## Section 2: Look inside the checkpoint

nnssl saves its checkpoints in nnU-Net's format: the model weights sit under a
`network_weights` key, next to optimizer state and training metadata. MIST
unwraps this automatically. Here we peek inside to see what we're loading. The
file is read with PyTorch's weights-only loader, so it cannot execute code.

In [ ]:
import torch

checkpoint = torch.load("nnfoundation_cnn.pth", map_location="cpu", weights_only=True)
print("top-level keys:", list(checkpoint))

weights = checkpoint["network_weights"]
encoder = {k: v for k, v in weights.items() if k.startswith("encoder.")}
stages = sorted({int(k.split(".")[2]) for k in encoder if k.startswith("encoder.stages.")})

print(f"tensors in the full network: {len(weights)}")
print(f"encoder tensors: {len(encoder)}  ({len(stages)} stages)")
# Each convolution is stored under two names that share one tensor, so count unique tensors.
unique = {v.data_ptr(): v.numel() for v in encoder.values()}
print(f"encoder parameters: {sum(unique.values()) / 1e6:.1f}M")
print("first stem convolution:", tuple(weights["encoder.stem.convs.0.conv.weight"].shape))

# nnssl also records how the model was pretrained.
plan = checkpoint["nnssl_adaptation_plan"]
pretrain = plan["pretrain_plan"]
setup = next(iter(pretrain["configurations"].values()))
print("architecture:", plan["architecture_plans"]["arch_class_name"])
print("pretraining patch size:", setup["patch_size"])
print("resampling:", setup["spacing_style"], "| normalization:", setup["normalization_schemes"])
print("median spacing of the pretraining data:", pretrain["original_median_spacing_after_transp"])

The stem convolution's shape, `(32, 1, 3, 3, 3)`, tells us the encoder takes a
**single input channel**. The pretraining plan shows that the volumes were
z-score normalized and **not resampled** (`noresample`): the model saw each scan
at its native resolution, in `192 × 192 × 192` patches, and the median spacing of
that data is 1 mm on every axis. Only the encoder transfers to MIST; the decoder
and the segmentation head are task-specific and start from random weights.

## Section 3: Convert, analyze, and configure

`mist_convert_msd` reorganizes the MSD dataset into MIST's layout and writes a
MIST `dataset.json`. `mist_analyze` then inspects the images and writes
`config.json` with defaults for spacing, patch size, normalization, model, and
training.

In [ ]:
!mist_convert_msd \
    --source Task09_Spleen \
    --output spleen_mist \
    --num-workers-conversion 2

In [ ]:
!mist_analyze \
    --data spleen_mist/dataset.json \
    --results spleen_results

In [ ]:
import json

config_path = "spleen_results/config.json"
with open(config_path) as f:
    config = json.load(f)

print("analyzer's target spacing:", config["spatial_config"]["target_spacing"])
print("analyzer's patch size    :", config["spatial_config"]["patch_size"])

### Match the run to the pretrained encoder

The analyzer picks a spacing and patch size for *this* dataset. Spleen CT has
thin in-plane voxels and thick slices, so its chosen spacing is anisotropic, and
for anisotropic spacing MIST's `resenc` (like nnU-Net) switches some kernels and
strides to 2D-like shapes along the thick axis. Those layers would no longer
match the pretrained ones.

Two settings make `resenc` match the pretrained network exactly:

- **`spatial_config.target_spacing` → `[1.0, 1.0, 1.0]`.** Isotropic spacing gives
  every stage 3 × 3 × 3 kernels and isotropic strides, as in nnFoundationCNN, and
  1 mm is the typical (median) spacing of its pretraining data.
- **`spatial_config.patch_size` → `[128, 128, 128]`.** Any patch of at least 128
  voxels per axis gives `resenc` the same six stages as the checkpoint, so the
  entire encoder transfers. (A smaller patch has fewer stages; MIST would load
  the ones that exist and warn that the deepest pretrained stage is unused.)

The rest is identical for both models:

- **`model.architecture` → `resenc`.**
- **`training.folds` → `[0]`** — train a single cross-validation fold (of five)
  instead of all of them.
- **`training.epochs` → `50`** with **`training.min_steps_per_epoch` → `250`** —
  12,500 optimizer steps per model. MIST's default is 1,000 epochs; this is a
  shortened schedule, but long enough for both models to learn the spleen.
- **`training.warmup_epochs` → `5`** — a linear learning-rate warmup over the
  first 10% of training (MIST's default of 20 epochs is sized for 1,000).
- **`inference.tta.enabled` → `false`** and **`patch_overlap` → `0.25`** — faster
  inference.

Both runs keep MIST's default learning rate and batch size (2 per GPU), so the
only difference between them is the starting weights. We also leave
`training.amp` alone: MIST trains in BF16 on GPUs that support it natively and
falls back to FP32 otherwise.

In [ ]:
# Match the pretrained encoder.
config["spatial_config"]["target_spacing"] = [1.0, 1.0, 1.0]
config["spatial_config"]["patch_size"] = [128, 128, 128]

# Model, a shortened training schedule, and faster inference (same for both runs).
config["model"]["architecture"] = "resenc"
config["training"]["folds"] = [0]
config["training"]["epochs"] = 50
config["training"]["min_steps_per_epoch"] = 250
config["training"]["warmup_epochs"] = 5
config["inference"]["tta"]["enabled"] = False
config["inference"]["inferer"]["params"]["patch_overlap"] = 0.25

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config["spatial_config"]))

### Preprocess

`mist_preprocess` resamples every scan to 1 mm, applies MIST's CT normalization
(clip to an intensity window, then z-score), and saves NumPy arrays to
`spleen_numpy/`. Both models train from these same arrays. nnFoundationCNN saw
z-scored intensities without clipping; fine-tuning adapts to the difference.
Resampling the thick slices to 1 mm makes the volumes large, so this step uses
every CPU core (`$(nproc)`): about 2.5 minutes on a G4 runtime.

In [ ]:
!mist_preprocess \
    --results spleen_results \
    --numpy spleen_numpy \
    --num-workers-preprocess $(nproc)

## Section 4: Train from scratch and from nnFoundationCNN

We copy the configured results directory twice, so the two runs share the same
config, folds, and preprocessed data and differ **only** in their starting
weights.

In [ ]:
import shutil

for run in ("spleen_scratch", "spleen_pretrained"):
    shutil.rmtree(run, ignore_errors=True)
    shutil.copytree("spleen_results", run)

First, the baseline: `resenc` from random initialization.

In [ ]:
!mist_train \
    --results spleen_scratch \
    --numpy spleen_numpy

Now the same run, starting from nnFoundationCNN. `--pretrained-weights` loads
every encoder tensor whose name and shape match, then MIST prints a **transfer
summary** before training starts. With the settings above it should report all
448 encoder tensors loaded and none skipped.

Two things to know about the summary and the warnings around it:

- **`--pretrained-config` is omitted.** For MIST or MISFIT checkpoints you would
  pass the source `config.json`, and MIST would check that patch size and spacing
  are compatible before loading. nnssl checkpoints don't have one, so MIST warns
  that it is skipping that check.
- **Channel strategy.** nnFoundationCNN has one input channel. For a multimodal
  dataset (say, four MRI sequences), MIST adapts the first convolution with
  `--input-channel-strategy` (by default it averages, then tiles across channels).
  Spleen CT has a single channel, so nothing is adapted here.

In [ ]:
!mist_train \
    --results spleen_pretrained \
    --numpy spleen_numpy \
    --pretrained-weights nnfoundation_cnn.pth

### Finalize both runs

Because we trained only fold 0 of 5, `mist_train` stops after training and
reminds us to finalize. `mist_finalize` evaluates the fold's held-out cases
(writing `results.csv`) and segments the test set.

In [ ]:
!mist_finalize --results spleen_scratch
!mist_finalize --results spleen_pretrained

## Section 5: Compare the two models

### Learning curves

MIST logs the training and validation loss after every epoch to TensorBoard
(`<results>/logs/fold_<k>/`). Plotting both runs together shows the effect of the
pretrained starting point.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator


def read_losses(results_dir, split):
    # add_scalars("losses", ...) writes each split to its own subdirectory.
    run_dir = next(Path(results_dir, "logs", "fold_0").glob(f"losses_{split}*"))
    events = EventAccumulator(str(run_dir))
    events.Reload()
    scalars = events.Scalars("losses")
    return [s.step for s in scalars], [s.value for s in scalars]


runs = {"from scratch": "spleen_scratch", "nnFoundationCNN": "spleen_pretrained"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, split in zip(axes, ("train", "validation"), strict=True):
    for label, results_dir in runs.items():
        ax.plot(*read_losses(results_dir, split), marker="o", label=label)
    ax.set_title(f"{split.capitalize()} loss")
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Dice + cross-entropy loss")
axes[1].legend()
plt.tight_layout()
plt.show()

### Cross-validation metrics

`results.csv` holds per-patient **Dice** (higher is better) and **HD95** (the
95th-percentile Hausdorff distance in mm, lower is better) for the fold's
held-out cases, followed by summary rows.

In [ ]:
import pandas as pd

summary_ids = ["Mean", "Std", "25th Percentile", "Median", "75th Percentile"]
scratch = pd.read_csv("spleen_scratch/results.csv").set_index("id")
pretrained = pd.read_csv("spleen_pretrained/results.csv").set_index("id")
metric_cols = [c for c in scratch.columns if c.endswith(("_dice", "_haus95"))]

comparison = pd.concat(
    {
        "from scratch": scratch.loc[summary_ids, metric_cols],
        "nnFoundationCNN": pretrained.loc[summary_ids, metric_cols],
    },
    axis=1,
)
comparison

### Rank the two runs with `mist_rank`

`mist_rank` ranks the runs case by case and metric by metric (1 = best), then
averages the ranks, so a **lower average rank is better**. `--significance-csv`
adds a pairwise Wilcoxon signed-rank test. With a single fold there are only
about nine held-out cases, so treat the p-value as indicative only.

In [ ]:
!mist_rank \
    --results spleen_scratch/results.csv spleen_pretrained/results.csv \
    --names scratch nnfoundation \
    --output-csv spleen_ranking.csv \
    --significance-csv spleen_significance.csv

In [ ]:
display(pd.read_csv("spleen_ranking.csv"))
display(pd.read_csv("spleen_significance.csv", index_col=0))

Keep the scale of this experiment in mind when reading the results: one fold,
about nine held-out cases, and a shortened schedule. The comparison shows how
the two starting points behave under identical training here, not a general
verdict on pretraining. A full evaluation would train all five folds for the
full schedule.

### Visualize

Finally, we look at one held-out case: the CT slice through the largest
cross-section of the spleen, with each model's prediction in red and the ground
truth outlined in cyan.

In [ ]:
import numpy as np
import SimpleITK as sitk
from matplotlib.colors import ListedColormap


def read_as_numpy(path):
    """Read a NIfTI file as a NumPy array in (x, y, z) axis order."""
    return sitk.GetArrayFromImage(sitk.ReadImage(str(path))).T


patient_id = scratch.index[~scratch.index.isin(summary_ids)][0]
case_dir = Path("spleen_mist/raw/train") / patient_id
ct = read_as_numpy(case_dir / "CT.nii.gz")
gt = read_as_numpy(case_dir / "mask.nii.gz")
z = int(np.argmax(gt.sum(axis=(0, 1))))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle(f"Held-out case {patient_id}, axial slice {z}", fontsize=12)
for ax, (label, results_dir) in zip(axes, runs.items(), strict=True):
    pred = read_as_numpy(Path(results_dir, "predictions/train/raw", f"{patient_id}.nii.gz"))
    dice = (scratch if results_dir == "spleen_scratch" else pretrained).loc[
        patient_id, metric_cols[0]
    ]
    ax.imshow(np.clip(ct[:, :, z].T, -160, 240), cmap="gray", origin="lower")
    overlay = np.ma.masked_where(pred[:, :, z] == 0, pred[:, :, z])
    ax.imshow(overlay.T, cmap=ListedColormap(["#ff2d2d"]), alpha=0.45, origin="lower")
    ax.contour(gt[:, :, z].T, levels=[0.5], colors="#00e5ff", linewidths=1.8)
    ax.set_title(f"{label}  (Dice {float(dice):.3f})")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Section 6: What's next

**nnFoundationViT.** nnssl also released
[nnFoundationViT](https://huggingface.co/MIC-DKFZ/nnFoundationViT), a 40-layer
transformer (PrimusX, about 550M encoder parameters) pretrained the same way.
Its encoder loads into MIST's `primus-x` architecture, which matches its layout
exactly. Loading it into a smaller Primus variant would use only part of the
encoder, and MIST warns if you try. Every patch dimension must be divisible by 8,
and its position embedding, pretrained at `192 × 192 × 192`, is resized
automatically for other patch sizes. It is several times larger than
nnFoundationCNN, so use a high-memory GPU:

```bash
wget https://huggingface.co/MIC-DKFZ/nnFoundationViT/resolve/main/checkpoint_final.pth -O nnfoundation_vit.pth
mist_train --results spleen_vit --numpy spleen_numpy \
    --model primus-x --pretrained-weights nnfoundation_vit.pth
```

**Pretrain on your own data.** [MISFIT](https://github.com/mist-medical/MISFIT)
is MIST's companion toolkit for self-supervised pretraining. It trains a MIST
encoder with a masked autoencoder on your unlabeled scans and writes weights that
`mist_train --pretrained-weights` loads directly. It can also start from
nnFoundationCNN (`misfit_train --init-weights`) to continue pretraining on your
own domain (MISFIT 0.2.0 and later), and its `misfit_embed` command turns any of these encoders into
per-scan embedding vectors for retrieval or classification.

**Train for real.** Raise `training.epochs`, train all five folds (remove
`training.folds`), and consider tuning the fine-tuning run's
`training.learning_rate` and `training.warmup_epochs` in its `config.json` (or
with `--learning-rate` and `--warmup-epochs`). nnssl doesn't publish recommended
fine-tuning values for MIST, so treat these as settings to experiment with. The
[transfer-learning documentation](https://mist-medical.readthedocs.io/en/latest/advanced_topics/#transfer-learning)
covers every option.